# Target encoding out of fold

A target encoding swaps a category for the average label of the training rows that carry it.
Computed on the same rows a model trains on, each row's own label sits inside its own feature. Here you build the
version that keeps every training row's label out of the features of its fold, then the
version used for new rows. NumPy only.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np

# Twelve support tickets: the product they are about, whether they were escalated, and a
# fold number that splits them into three folds.
product = np.array(["app", "web", "app", "app", "web", "api", "cli", "app", "web", "web", "app", "app"])
escalated = np.array([1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1])
fold = np.array([0, 1, 2, 0, 1, 2, 0, 1, 2, 0, 1, 2])
for f in range(3):
    print(f"fold {f}:", list(zip(product[fold == f], escalated[fold == f])))

## Part 1 (coding task): Encode the training rows out of fold

Write `encode_out_of_fold(cats, y, fold, alpha)`, which returns one float per row. For a row
in fold $f$, use only the rows in the **other** folds:

- the prior $p$ is the mean of $y$ over the other folds.
- if those rows contain $n$ rows of the row's category, with labels adding up to $s$, the
  encoding is
  $$\frac{s + \alpha p}{n + \alpha},$$
  which shrinks small categories toward $p$ and gives exactly $p$ to a category the other
  folds do not have (for $\alpha > 0$).

In [ ]:
def encode_out_of_fold(cats, y, fold, alpha):
    """Each row's encoding, computed from the rows of the other folds only."""
    # TODO
    raise NotImplementedError

In [ ]:
enc = np.asarray(encode_out_of_fold(product, escalated, fold, alpha=2.0), dtype=float)
assert enc.shape == (12,), f"expected one value per row, got shape {enc.shape}"
assert not np.isnan(enc).any(), "some encodings are nan: a category the other folds lack should get the prior"
for i in range(12):
    flipped = escalated.copy()
    flipped[i] = 1 - flipped[i]
    moved = np.asarray(encode_out_of_fold(product, flipped, fold, alpha=2.0), dtype=float) != enc
    same_fold = fold == fold[i]
    assert not (moved & same_fold).any(), (
        f"changing the label of row {i} (fold {fold[i]}) changed the encoding of rows "
        f"{np.flatnonzero(moved & same_fold).tolist()} in the same fold: those must come from the other folds only")
assert np.isclose(enc[6], 4 / 8), f"row 6 is the only cli ticket, so it should get the prior of folds 1 and 2, 0.5, got {enc[6]}"
expected = np.array([2 / 3, 9 / 16, 13 / 24, 2 / 3, 9 / 16, 5 / 8, 1 / 2, 17 / 24, 13 / 20, 2 / 5, 17 / 24, 13 / 24])
assert np.allclose(enc, expected), f"encodings {enc.round(4)}, expected {expected.round(4)}"
plain = np.asarray(encode_out_of_fold(product, escalated, fold, alpha=0.0), dtype=float)
assert np.isclose(plain[0], 3 / 4), f"with alpha = 0 an app ticket in fold 0 gets the plain mean of the app tickets in folds 1 and 2, 3/4, got {plain[0]}"
print("Part 1 passed")

## Part 2 (coding task): Encode new rows

New tickets are encoded with every training row. Write
`encode_new(train_cats, train_y, new_cats, alpha)`, which returns one float per new row:
$(s + \alpha p)/(n + \alpha)$ with $p$ the mean of all training labels and $s$, $n$ the label
sum and count of the category among all training rows. A category with no training rows gets
$p$, even when $\alpha = 0$.

In [ ]:
def encode_new(train_cats, train_y, new_cats, alpha):
    """Encodings for new rows, from all the training rows."""
    # TODO
    raise NotImplementedError

In [ ]:
new = np.array(["web", "cli", "desktop", "app"])
enc_new = np.asarray(encode_new(product, escalated, new, alpha=2.0), dtype=float)
assert enc_new.shape == (4,), f"expected one value per new row, got shape {enc_new.shape}"
assert not np.isnan(enc_new).any(), "desktop never appears in training: it should get the prior, not nan"
assert np.isclose(enc_new[2], 7 / 12), f"desktop should get the mean of all 12 training labels, 7/12, got {enc_new[2]}"
expected = np.array([(2 + 2 * 7 / 12) / 6, (1 + 2 * 7 / 12) / 3, 7 / 12, (4 + 2 * 7 / 12) / 8])
assert np.allclose(enc_new, expected), f"encodings {enc_new.round(4)}, expected {expected.round(4)}"
zero = np.asarray(encode_new(product, escalated, new, alpha=0.0), dtype=float)
assert np.allclose(zero, [0.5, 1.0, 7 / 12, 2 / 3]), f"with alpha = 0: {zero.round(4)}, expected plain means and the prior for desktop"
print("Part 2 passed")

## Solution

### Part 1: Encode the training rows out of fold

In [ ]:
def encode_out_of_fold(cats, y, fold, alpha):
    """Each row's encoding, computed from the rows of the other folds only."""
    y = np.asarray(y, dtype=float)
    out = np.empty(len(y))
    for f in np.unique(fold):
        inside, outside = fold == f, fold != f
        prior = y[outside].mean()
        for c in np.unique(cats[inside]):
            rows = outside & (cats == c)
            out[inside & (cats == c)] = (y[rows].sum() + alpha * prior) / (rows.sum() + alpha)
    return out

**Why it works.** The outer loop holds out one fold at a time, and every number used for its
rows, the prior and each category's $s$ and $n$, is computed from `outside`. Changing a label
in fold $f$ can then move only the encodings of the other folds, which is what the check
tries for all twelve rows. The two encodings students often write first both fail it. The
plain mean over all rows puts each row's own label into its value. Leaving only the row
itself out keeps its own label away, but the other tickets of its fold are still in, so the
two app tickets of fold 0 would encode each other's labels. Here the app tickets of folds 1
and 2 have 3 escalations in 4, and those folds have prior $4/8$, so with $\alpha = 2$ an app
ticket of fold 0 gets $(3 + 2 \cdot 0.5)/(4 + 2) = 2/3$, pulled from $3/4$ toward the
prior. The single cli ticket finds no cli rows in the other folds and gets the prior,
$1/2$.

### Part 2: Encode new rows

In [ ]:
def encode_new(train_cats, train_y, new_cats, alpha):
    """Encodings for new rows, from all the training rows."""
    train_y = np.asarray(train_y, dtype=float)
    prior = train_y.mean()
    out = np.full(len(new_cats), prior)
    for c in np.unique(new_cats):
        rows = train_cats == c
        if rows.any():
            out[new_cats == c] = (train_y[rows].sum() + alpha * prior) / (rows.sum() + alpha)
    return out

**Why it works.** New rows have no label to leak, so every training row can be used, and the
same shrinkage formula keeps a category seen once, cli with a single escalated ticket, from
being encoded as a certain escalation: with $\alpha = 2$ it gets $(1 + 2 \cdot 7/12)/3 \approx 0.72$
instead of 1. A category the training rows never had, desktop, starts from `prior` and is
never overwritten, so it needs no special case even at $\alpha = 0$, where the formula would
divide 0 by 0.